# 13 — The complete decision assistant
This is the same Crew and Flow from earlier lessons. The CLI supplies one
request, a customer identity, and an explicit approval flag for a local mock
email. No external email service is connected.



## Problem

The pieces must work together behind one entry point.


## Concept

The CLI invokes the existing CrewAI Flow; it does not duplicate decision logic.

The interface passes a request into one application entry point. Memory enriches context, the Flow owns control, the Crew researches, and the action gate controls the only side effect.


## Abstraction

question/customer/approval → Flow → Crew → report/action route. This is the smallest new layer introduced by this lesson; the earlier layers remain in place.


## Architecture

```text
CLI request ──▶ run_assistant ──▶ scoped Memory
                    │                  │
                    ▼                  ▼
                DecisionFlow ──▶ Research Crew ──▶ DecisionReport
                    │
                    └──▶ approval gate ──▶ local mock action
```

Follow the arrows while executing the cells. A tool result or validation failure is an observation, not an instruction that the model may bypass.


## Step 1 — Parse the CLI request and reject an approval flag without an action request

Parse the CLI request and reject an approval flag without an action request.


In [ ]:
import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()))

from workshop.app import run_assistant

parser = argparse.ArgumentParser()
parser.add_argument("--question", default="Should Northstar AI pilot Meridian AI?")
parser.add_argument("--customer", default="customer-a")
parser.add_argument("--remember", help="Store a customer constraint before this run")
parser.add_argument("--request-email", action="store_true")
parser.add_argument("--approve-email", action="store_true")
args, _ = parser.parse_known_args()
if args.approve_email and not args.request_email:
    parser.error("--approve-email requires --request-email")



## Run the end-to-end Flow
Inspect the recommendation, the citation IDs, and the selected route.
Try --request-email first, then explicitly add --approve-email.



## Step 2 — Run the same Flow used earlier

Run the same Flow used earlier; inspect report and action route separately.


In [ ]:
flow, output = run_assistant(
    question=args.question,
    customer_id=args.customer,
    remember=args.remember,
    request_email=args.request_email,
    approved=args.approve_email,
)
print(output)
print("Report:", flow.state.report)
print("Action:", flow.state.action_result or "none")


## Execution flow

question/customer/approval → Flow → Crew → report/action route. Read the printed result together with the task, selected tool, state, or trace. If the result differs, locate the first boundary where the observed value changed.


## Failure mode to know about

A good final report does not imply the action route is safe. Exercise the no-approval path, approved path, and replay of the same action ID separately.


## Production implication

The CLI is the workshop's minimal interface; an API or UI can call the same entry point later. Do not reimplement the Flow's approval logic in that interface.


## Your turn

Run a report, then request a mock email without and with approval.

**Checkpoint:** The report has sources and only explicit approval records the mock action.
